## Qualitative Message Sampler

Randomly samples messages from a specified sub-category and displays the full `truncated_content`, the LLM reasoning for that label, and any co-occurring sub-categories. Used for manual qualitative inspection to develop and validate finding narratives.

In [1]:
import pandas as pd

df_classifications = pd.read_csv(
    "../data/classifications/classifications_for_analysis.csv"
)


def sample_category(df, sub_category, sample_n=40, random_seed=42):
    target_rows = df[df["sub_category"] == sub_category]
    target_msgs = target_rows[["sha", "index_in_chat"]].drop_duplicates()

    if len(target_msgs) < sample_n:
        sampled = target_msgs
    else:
        sampled = target_msgs.sample(sample_n, random_state=random_seed)

    results = []
    for _, (sha, idx) in sampled[["sha", "index_in_chat"]].iterrows():
        msg_rows = df[(df["sha"] == sha) & (df["index_in_chat"] == idx)]
        content = msg_rows["truncated_content"].iloc[0]
        target_row = msg_rows[msg_rows["sub_category"] == sub_category]
        reasoning = (
            target_row["reasoning"].iloc[0] if len(target_row) > 0 else "(not found)"
        )
        results.append(
            {
                "sha": sha,
                "index_in_chat": idx,
                "content": content,
                "sub_categories": msg_rows["sub_category"].tolist(),
                "reasoning": reasoning,
            }
        )

    return {
        "sub_category": sub_category,
        "total_found": len(target_msgs),
        "sampled_n": len(sampled),
        "messages": results,
    }


def print_sample_result(result):
    print(f"Showing {result['sampled_n']} messages for: [{result['sub_category']}]")
    print("=" * 80)
    for msg in result["messages"]:
        print(f"[sha: {msg['sha']}  msg #{msg['index_in_chat']}]")
        print(msg["content"])
        print(msg["sub_categories"])
        print(msg["reasoning"])
        print("=" * 80)


def sample_all_categories(df, sample_n=40, random_seed=42):
    all_messages = []
    for sub_cat in sorted(df["sub_category"].unique()):
        result = sample_category(
            df, sub_cat, sample_n=sample_n, random_seed=random_seed
        )
        for msg in result["messages"]:
            msg["sampled_from_category"] = sub_cat
            all_messages.append(msg)

    return pd.DataFrame(all_messages)

In [2]:
df_sampled = sample_all_categories(df_classifications, sample_n=40, random_seed=42)
print(
    f"{len(df_sampled)} messages sampled across {df_sampled['sampled_from_category'].nunique()} categories"
)
df_sampled.to_csv("results/qualitative_samples_42.csv", index=False)

800 messages sampled across 20 categories


In [3]:
result = sample_category(
    df_classifications, "2.3 Error Persistence", sample_n=40, random_seed=42
)
# print_sample_result(result)